# 08 · MENA Complex Emergencies — Cross-Regional Bubble Map

This notebook maps the attached MENA complex emergencies table as **proportional bubbles** on Mapbox tiles.  
No choropleth is used. Bubble size represents allocation amount, and the country details are shown **directly on the map** beside each bubble.

Styling follows the Mapbox configuration pattern from notebook 7.


In [29]:
import json
import math
import warnings
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings('ignore', category=DeprecationWarning, module=r'plotly\..*')
warnings.filterwarnings('ignore', category=DeprecationWarning, message=r'.*[Ss]cattermapbox.*deprecated.*')

# -- Paths & Mapbox configuration ---------------------------------------------
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
CONFIG_DIR = DATA_DIR / 'config'
OUTPUTS_DIR = ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

mapbox_cfg = json.loads((CONFIG_DIR / 'mapbox.local.json').read_text(encoding='utf-8'))
MAPBOX_TOKEN = mapbox_cfg['mapbox_token']
MAPBOX_STYLE = mapbox_cfg['style_url']


def format_chf_compact(value: float) -> str:
    if value >= 1_000_000:
        short = value / 1_000_000
        return f"CHF {short:.1f}M" if short % 1 else f"CHF {short:.0f}M"
    if value >= 1_000:
        short = value / 1_000
        return f"CHF {short:.1f}k" if short % 1 else f"CHF {short:.0f}k"
    return f"CHF {value:,.0f}"


# -- Attached table transcribed to structured data -----------------------------
# label_lon/lat = centre of the callout box
# label_width_deg / label_height_deg = visual size of box (degrees)
# Positions chosen so no two boxes overlap each other.
rows = [
    {
        'country': 'Iran',
        'modality': 'EA (Loan)',
        'allocation_chf': 1_525_304,
        'request_date': '28/Feb',
        'approval_date': '28/Feb',
        'longitude': 53.6880,
        'latitude': 32.4279,
        'label_lon': 58.5,
        'label_lat': 35.5,
        'label_width_deg': 10.0,
        'label_height_deg': 3.2,
    },
    {
        'country': 'Lebanon',
        'modality': 'EA (Loan)',
        'allocation_chf': 2_000_000,
        'request_date': '6/Mar',
        'approval_date': '6/Mar',
        'longitude': 35.8623,
        'latitude': 33.8547,
        'label_lon': 26.5,
        'label_lat': 31.5,
        'label_width_deg': 10.5,
        'label_height_deg': 3.2,
    },
    {
        'country': 'Armenia',
        'modality': 'I-DREF',
        'allocation_chf': 80_000,
        'request_date': '3/Mar',
        'approval_date': '4/Mar',
        'longitude': 44.5152,
        'latitude': 40.1872,
        'label_lon': 38.0,
        'label_lat': 44.5,
        'label_width_deg': 9.0,
        'label_height_deg': 2.9,
    },
    {
        'country': 'Iraq',
        'modality': 'I-DREF',
        'allocation_chf': 80_000,
        'request_date': '5/Mar',
        'approval_date': '5/Mar',
        'longitude': 43.6793,
        'latitude': 33.2232,
        'label_lon': 42.0,
        'label_lat': 29.0,
        'label_width_deg': 8.5,
        'label_height_deg': 2.9,
    },
    {
        'country': 'Azerbaijan',
        'modality': 'I-DREF',
        'allocation_chf': 90_800,
        'request_date': '6/Mar',
        'approval_date': '7/Mar',
        'longitude': 47.5769,
        'latitude': 40.1431,
        'label_lon': 52.6,
        'label_lat': 43.0,
        'label_width_deg': 9.5,
        'label_height_deg': 2.9,
    },
    {
        'country': 'Turkmenistan',
        'modality': 'I-DREF',
        'allocation_chf': 90_800,
        'request_date': '13/Mar',
        'approval_date': '16/Mar',
        'longitude': 59.5563,
        'latitude': 38.9697,
        'label_lon': 67.0,
        'label_lat': 42.5,
        'label_width_deg': 9.5,
        'label_height_deg': 2.9,
    },
]

mena_df = pd.DataFrame(rows)

# Bubble area uses sqrt scaling so the largest loan does not dominate the map.
max_alloc = mena_df['allocation_chf'].max()
mena_df['bubble_size'] = mena_df['allocation_chf'].apply(
    lambda value: 22 + 52 * math.sqrt(value / max_alloc)
)

mena_df['label_text'] = mena_df.apply(
    lambda row: (
        f"{row['country']}<br>"
        f"{row['modality']}  |  {format_chf_compact(row['allocation_chf'])}<br>"
        f"Req {row['request_date']}  |  Appr {row['approval_date']}"
    ),
    axis=1,
)

COLOR_MAP = {
    'EA (Loan)': '#FF3B30',
    'I-DREF': '#B4612D',
}

print(f"Loaded {len(mena_df)} records")
print(f"Total allocation: CHF {mena_df['allocation_chf'].sum():,.0f}")
mena_df


Loaded 6 records
Total allocation: CHF 3,866,904


,country,modality,allocation_chf,request_date,approval_date,longitude,latitude,label_lon,label_lat,label_width_deg,label_height_deg,bubble_size,label_text
0,Iran,EA (Loan),1525304,28/Feb,28/Feb,53.6880,32.4279,58.5,35.5,10.0,3.2,67.411574,Iran<br>EA (Loan) | CHF 1.5M<br>Req 28/Feb ...
1,Lebanon,EA (Loan),2000000,6/Mar,6/Mar,35.8623,33.8547,26.5,31.5,10.5,3.2,74.000000,Lebanon<br>EA (Loan) | CHF 2M<br>Req 6/Mar ...
2,Armenia,I-DREF,80000,3/Mar,4/Mar,44.5152,40.1872,38.0,44.5,9.0,2.9,32.400000,Armenia<br>I-DREF | CHF 80k<br>Req 3/Mar | ...
3,Iraq,I-DREF,80000,5/Mar,5/Mar,43.6793,33.2232,42.0,29.0,8.5,2.9,32.400000,Iraq<br>I-DREF | CHF 80k<br>Req 5/Mar | Ap...
4,Azerbaijan,I-DREF,90800,6/Mar,7/Mar,47.5769,40.1431,52.6,43.0,9.5,2.9,33.079783,Azerbaijan<br>I-DREF | CHF 90.8k<br>Req 6/Ma...
5,Turkmenistan,I-DREF,90800,13/Mar,16/Mar,59.5563,38.9697,67.0,42.5,9.5,2.9,33.079783,Turkmenistan<br>I-DREF | CHF 90.8k<br>Req 13...


In [30]:
def label_box_geometry(center_lon, center_lat, width_deg, height_deg):
    cos_lat = max(math.cos(math.radians(center_lat)), 0.35)
    half_width_lon = (width_deg / cos_lat) / 2
    half_height_lat = height_deg / 2
    return half_width_lon, half_height_lat


def make_label_box_trace(center_lon, center_lat, width_deg, height_deg):
    half_width_lon, half_height_lat = label_box_geometry(
        center_lon, center_lat, width_deg, height_deg
    )

    lons = [
        center_lon - half_width_lon,
        center_lon + half_width_lon,
        center_lon + half_width_lon,
        center_lon - half_width_lon,
        center_lon - half_width_lon,
    ]
    lats = [
        center_lat - half_height_lat,
        center_lat - half_height_lat,
        center_lat + half_height_lat,
        center_lat + half_height_lat,
        center_lat - half_height_lat,
    ]

    return go.Scattermapbox(
        lon=lons,
        lat=lats,
        mode='lines',
        fill='toself',
        fillcolor='rgba(248,244,235,0.97)',
        line=dict(color='#b9b2a6', width=1.3),
        hoverinfo='skip',
        showlegend=False,
        name='',
    )


def connector_endpoint(row):
    half_width_lon, half_height_lat = label_box_geometry(
        row['label_lon'], row['label_lat'], row['label_width_deg'], row['label_height_deg']
    )

    delta_lon = row['longitude'] - row['label_lon']
    delta_lat = row['latitude'] - row['label_lat']

    if abs(delta_lon) < 1e-9 and abs(delta_lat) < 1e-9:
        return row['label_lon'], row['label_lat']

    scale = max(abs(delta_lon) / half_width_lon, abs(delta_lat) / half_height_lat)
    return row['label_lon'] + delta_lon / scale, row['label_lat'] + delta_lat / scale


def make_connector_trace(row):
    anchor_lon, anchor_lat = connector_endpoint(row)
    return go.Scattermapbox(
        lon=[row['longitude'], anchor_lon],
        lat=[row['latitude'], anchor_lat],
        mode='lines',
        line=dict(color='rgba(90,90,90,0.6)', width=1.2),
        hoverinfo='skip',
        showlegend=False,
        name='',
    )


def make_label_text_trace(row):
    return go.Scattermapbox(
        lon=[row['label_lon']],
        lat=[row['label_lat']],
        mode='text',
        text=[row['label_text']],
        textposition='middle center',
        textfont=dict(size=13, color='#17202a'),
        hoverinfo='skip',
        showlegend=False,
        name='',
    )


def build_mena_complex_emergency_map(width=980, height=620):
    background_traces = []
    bubble_traces = []
    text_traces = []

    for _, row in mena_df.iterrows():
        background_traces.append(
            make_label_box_trace(
                center_lon=row['label_lon'],
                center_lat=row['label_lat'],
                width_deg=row['label_width_deg'],
                height_deg=row['label_height_deg'],
            )
        )
        background_traces.append(make_connector_trace(row))
        text_traces.append(make_label_text_trace(row))

    for modality, subset in mena_df.groupby('modality', sort=False):
        bubble_traces.append(
            go.Scattermapbox(
                lon=subset['longitude'],
                lat=subset['latitude'],
                mode='markers',
                marker=dict(
                    size=subset['bubble_size'],
                    color=COLOR_MAP[modality],
                    opacity=0.9,
                    sizemode='diameter',
                ),
                name=modality,
                hoverinfo='skip',
            )
        )

    fig = go.Figure(data=background_traces + bubble_traces + text_traces)
    fig.update_layout(
        mapbox=dict(
            accesstoken=MAPBOX_TOKEN,
            style=MAPBOX_STYLE,
            center=dict(lon=47.0, lat=35.2),
            zoom=3.55,
            pitch=0,
        ),
        title=dict(
            text=(
                f"<b>DREF · MENA Complex Emergencies</b>  |  Cross-Regional Response<br>"
                f"<sup>Total: CHF {mena_df['allocation_chf'].sum():,.0f}  ·  "
                f"Bubble size ∝ allocation amount  ·  Text boxes show funding, allocation, request and approval</sup>"
            ),
            x=0.5,
            font=dict(size=15),
        ),
        legend=dict(
            title='Modality',
            orientation='v',
            x=0.01,
            y=0.99,
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#cccccc',
            borderwidth=1,
        ),
        margin=dict(l=0, r=0, t=65, b=20),
        width=width,
        height=height,
        paper_bgcolor='white',
        plot_bgcolor='white',
    )
    return fig


fig_mena_complex = build_mena_complex_emergency_map()
fig_mena_complex.show()
print('Boxes enlarged, repositioned to avoid overlap, bubbles and text scaled up.')


Boxes enlarged, repositioned to avoid overlap, bubbles and text scaled up.
